# Exploring Architect

This notebook demonstrates how to use the `Architect` class to load a manifest and run dataflows.
The example manifest (`examples/example/manifest.yaml`) models a cat-happiness device with requirements and solutions.

In [ ]:
import ibis

from iacs.architect import Architect

ibis.options.interactive = True

## Load the Example Manifest

`Architect.from_manifest` reads the YAML files and builds the registry. Then call `load_dataflow` with a module name string to attach Hamilton dataflow modules from `iacs.dataflows`.

In [ ]:
architect = Architect.from_manifest("../examples/example")

# architect.load_dataflow("audit.traceability")
# architect.load_dataflow("audit.todo")
# architect.load_dataflow("audit.requirement_coverage")
architect.load_dataflow("export_manifest")

## Inspect the Registry

The registry holds the loaded data in component-centered format — one table per component type.

In [ ]:
# Component types present in the manifest
architect.registry.component_types

In [ ]:
# View a specific component table
architect.registry.view("description")

In [ ]:
architect.registry.view("requirement")

## Explore the Dataflows

The dataflows attached to the architect are Hamilton DAGs. You can visualize the full graph of available computations.

In [ ]:
# List all computable outputs (non-input nodes in the DAG)
architect.outputs

In [ ]:
# Visualize the full dataflow DAG
architect._driver.display_all_functions(orient="TB")

## Execute Dataflows

Call `architect.execute` with a list of output names to compute results.

In [ ]:
# Run the traceability audit — entities not linked to any requirement show up here
results = architect.execute(["traceability"])
results["traceability"]

In [ ]:
# Run the todo audit — outstanding todos in the manifest
results = architect.execute(["todo"])
results["todo"]

In [ ]:
# Run multiple outputs at once
results = architect.execute(["all_entities", "req_entities", "solution_entities", "orphan_entities"])
for name, table in results.items():
    print(f"--- {name} ---")
    print(table.execute())
    print()

## Visualize Execution Paths

You can also visualize the subgraph that Hamilton will execute for a specific output.

In [ ]:
architect._driver.visualize_execution(["traceability"])